# Full pipeline — Colab entrypoint

Runs `core.pipeline.run_job` end to end (script -> voice -> lip-sync -> optional
captions -> assembled HD MP4), in a single Colab runtime.

Colab has one Python environment, not the isolated per-stage venvs used
locally (`core/envs.py`) — so this cell installs every stage's requirements
into the same runtime. That works here because Colab starts from a clean
image each session; it would not work as a persistent local setup, which is
why `scripts/setup_envs.sh` isolates them there instead.

Real presenter media requires `CONSENT.md` to read `STATUS: GRANTED` in the
repo you clone here — this notebook runs the same consent gate as the CLI.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
REPO_URL = "<your fork/clone URL>"
REPO_DIR = "/content/presenter-video"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/presenter-video'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['PRESENTER_DRIVE_DIR'] = DRIVE_DIR
os.environ['PRESENTER_WEIGHTS_DIR'] = f'{DRIVE_DIR}/weights'
os.environ['HF_HOME'] = f'{DRIVE_DIR}/weights/hf'
os.environ['TORCH_HOME'] = f'{DRIVE_DIR}/weights/torch'
# Point the orchestrator's per-stage lookup at *this* interpreter for every
# stage, since Colab has one environment, not core/envs.py's isolated venvs.
import sys
for stage in ("VOICE", "LIPSYNC", "CAPTIONS"):
    os.environ[f'PRESENTER_{stage}_PYTHON'] = sys.executable

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements/voice.txt
!pip install -q -r requirements/lipsync.txt
# !pip install -q -r requirements/captions.txt  # only if you're turning captions on
!apt -y install libgl1 > /dev/null
!bash scripts/vendor_latentsync.sh

### Job config

Uses the openly-licensed placeholders in `assets/samples/` by default — no
consent needed to run this cell. To use a real presenter, copy
`configs/example_job.yaml`, point it at real assets under e.g.
`assets/presenter/`, complete `CONSENT.md` in the cloned repo, and load that
config path instead.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from core.config import load_config
from core.pipeline import run_job

cfg = load_config(f"{REPO_DIR}/configs/selftest.yaml")
cfg.raw["runtime"]["drive_dir"] = DRIVE_DIR

final_path = run_job(cfg)
print("done ->", final_path)

In [ ]:
from IPython.display import Video
Video(str(final_path), embed=True, width=360)